Before running the file Upload all your data set on your goole drive in a zip format

In [1]:
#To get the average frame count 
import json
import glob
import numpy as np
import cv2
import copy
#change the path accordingly
video_files =  glob.glob('./real_yt/*.mp4')
frame_count = []
for video_file in video_files:
  cap = cv2.VideoCapture(video_file)
  if(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))<150):
    video_files.remove(video_file)
    continue
  frame_count.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("frames" , frame_count)
print("Total number of videos: " , len(frame_count))
print('Average frame per video:',np.mean(frame_count))

frames [396, 460, 693, 303, 309, 385, 310, 505, 637, 588, 474, 632, 363, 332, 676, 437, 678, 398, 432, 1385, 390, 431, 325, 553, 306, 747, 322, 315, 434, 416, 297, 482, 310, 809, 407, 423, 337, 454, 305, 609, 548, 349, 338, 495, 335, 350, 504, 319, 485, 360, 530, 717, 323, 417, 368, 394, 373, 431, 669, 721, 844, 337, 423, 930, 402, 442, 435, 538, 401, 774, 367, 453, 311, 703, 345, 613, 329, 611, 523, 424, 483, 1170, 316, 545, 375, 329, 935, 309, 665, 751, 687, 639, 460, 308, 559, 1006, 504, 478, 299, 613, 423, 1148, 389, 595, 462, 300, 356, 556, 300, 298, 565, 433, 418, 841, 1269, 320, 327, 548, 352, 502, 405, 553, 309, 514, 318, 500, 328, 864, 735, 330, 662, 306, 413, 767, 346, 327, 611, 606, 410, 378, 485, 508, 962, 414, 344, 475, 1003, 667, 475, 363, 456, 347, 313, 732, 594, 829, 518, 486, 444, 325, 534, 369, 685, 304, 1021, 387, 756, 341, 1228, 649, 504, 541, 752, 721, 414, 504, 376, 410, 521, 1260, 569, 478, 620, 390, 1254, 875, 334, 986, 310, 512, 331, 372, 287, 372, 567, 333, 42

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
# You can also check how many GPUs and their names:
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce MX450


In [3]:
import os
import glob
import cv2
import numpy as np
from tqdm.auto import tqdm

def frame_extract(path, max_frames=150):
    """Yield up to max_frames BGR frames (as uint8 arrays)."""
    vid = cv2.VideoCapture(path)
    count = 0
    while count < max_frames:
        success, frame = vid.read()
        if not success or frame is None:
            break
        yield frame
        count += 1
    vid.release()

def create_face_videos(video_paths, out_dir):
    """Use OpenCV Haar cascade to crop faces and save 112×112 MP4s."""
    os.makedirs(out_dir, exist_ok=True)
    cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    
    for src in tqdm(video_paths, desc="Videos"):
        dst = os.path.join(out_dir, os.path.basename(src))
        if os.path.exists(dst):
            continue
        
        writer = cv2.VideoWriter(dst, fourcc, 30, (112,112), True)
        frames_written = 0
        
        for frame in frame_extract(src):
            # detect faces in grayscale
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = cascade.detectMultiScale(
                gray,
                scaleFactor=1.1,
                minNeighbors=5,
                minSize=(50,50)
            )
            if len(faces) == 0:
                continue
            
            # take the first face
            x,y,w,h = faces[0]
            face = frame[y:y+h, x:x+w]
            # resize & write
            face = cv2.resize(face, (112,112))
            writer.write(face)
            frames_written += 1
            
            # stop once we have 150 frames
            if frames_written >= 150:
                break
        
        writer.release()
        print(f"Saved: {dst} ({frames_written} frames)")


In [4]:
video_files = glob.glob('./real_yt/*.mp4')  # Adjust path as needed
create_face_videos(video_files, './real_yt_face_videos/')

Videos:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: ./real_yt_face_videos/030.mp4 (150 frames)
Saved: ./real_yt_face_videos/241.mp4 (150 frames)


In [5]:
video_files = glob.glob('./fake_yt/*.mp4')
create_face_videos(video_files, './fake_yt_face_videos/')

Videos:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: ./fake_yt_face_videos/014_790.mp4 (150 frames)
Saved: ./fake_yt_face_videos/286_267.mp4 (150 frames)
